# Pandas — Phase 3: Data Cleaning & Preprocessing
### Credit Card Risk Analysis Track

**Topics in this phase:**
11. Identifying Missing Data — `.isnull()`, `.isna()`, `.sum()`
12. Dropping Missing Data — `.dropna()`
13. Imputing Missing Data — `.fillna()`
14. Handling Duplicates — `.duplicated()`, `.drop_duplicates()`
15. Data Type Casting — `.astype()`, `pd.to_numeric`

**Dataset:** this phase uses a **new, messier file**: `loan_applications_raw.csv` — same 500 customers as before, but with realistic problems layered in: a text-based `interest_rate` column (e.g. `"5.72%"`), an entirely empty `internal_notes` column, 8 exact duplicate rows, and 13 duplicate `application_id`s from re-submitted applications. Place it in the same folder as this notebook.

**How to use this notebook:**
- Each question has a `YOUR CODE HERE` cell — attempt it first.
- The `Solution` cell right after shows one correct approach — compare, don't just copy.
- All solutions were run against the actual dataset before this notebook was assembled.

## Setup

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("loan_applications_raw.csv", parse_dates=["application_date"])
print(df.shape)
df.head()

## Topic 11: Identifying Missing Data

You can't fix what you haven't measured. Always start here.

**Q1.** Get a full boolean grid of where values are missing using `df.isnull()`, into `missing_mask`. Print its shape (it should match `df`'s shape exactly).

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
missing_mask = df.isnull()
print(missing_mask.shape)

**Q2.** Get the count of missing values **per column** into `missing_per_column`, chaining `.isnull().sum()`.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
missing_per_column = df.isnull().sum()
print(missing_per_column)

**Q3.** Get just the missing count for `annual_income` into `missing_income_count` (`.isna()` is an exact alias for `.isnull()` — both are used interchangeably in practice).

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
missing_income_count = df["annual_income"].isna().sum()
print(missing_income_count)

**Q4.** Get the **total** number of missing cells across the entire DataFrame into `total_missing_cells` (chain `.sum()` twice — once per column, once across columns).

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
total_missing_cells = df.isnull().sum().sum()
print(total_missing_cells)

**Q5.** Raw counts don't tell the whole story on their own — compute the **percentage** missing per column into `missing_pct` (missing count / total rows * 100, rounded to 2 decimals). Which column is completely useless because of this?

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)
print(missing_pct)

**Q6.** Get every row that has **at least one** missing value anywhere, into `rows_with_any_missing`, using `.isnull().any(axis=1)` as a boolean mask. (Given what you found in Q5, think about why this returns basically the whole dataset — you'll fix that in Topic 12.)

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
rows_with_any_missing = df[df.isnull().any(axis=1)]
print(rows_with_any_missing.shape)

## Topic 12: Dropping Missing Data

Dropping is the simplest fix — but it's also the easiest way to silently throw away most of your dataset if you're not careful, as you're about to see.

**Q7.** Drop any column that is **entirely** empty using `.dropna(axis=1, how="all")`, into `df_no_empty_cols`. Confirm `internal_notes` is gone and print the new shape.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
df_no_empty_cols = df.dropna(axis=1, how="all")
print("internal_notes" in df_no_empty_cols.columns, df_no_empty_cols.shape)

**Q8.** Now try `df.dropna()` with **no arguments** on the original `df`, into `df_complete_rows`. Print its shape. (This drops any row with *any* missing value in *any* column — including the fully-empty `internal_notes` column, which means it wipes out every single row. This is exactly why Q7 has to happen first in a real cleaning pipeline.)

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
df_complete_rows = df.dropna()
print(df_complete_rows.shape)

**Q9.** Be more targeted: drop rows only where `annual_income` specifically is missing, using `subset=["annual_income"]`, into `df_income_required`. Print the shape.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
df_income_required = df.dropna(subset=["annual_income"])
print(df_income_required.shape)

**Q10.** Use `thresh=12` to keep only rows that have **at least 12 non-null values** out of the 14 columns (a more forgiving rule than requiring zero missing values), into `df_thresh`. Print the shape.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
df_thresh = df.dropna(thresh=12)
print(df_thresh.shape)

**Q11.** Combine what you learned: build `df_clean_step1` by first dropping the fully-empty column(s), **then** dropping any remaining row missing `annual_income` or `credit_score` (chain the two `.dropna()` calls). Print the shape.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
df_clean_step1 = df.dropna(axis=1, how="all").dropna(subset=["annual_income", "credit_score"])
print(df_clean_step1.shape)

## Topic 13: Imputing Missing Data

Dropping loses data. Imputing keeps the row and fills the gap with a defensible business-logic value instead.

**Q12.** Make a copy `df_imputed = df.copy()`. Compute the median of `annual_income`, then use `.fillna()` to replace missing incomes with that median, overwriting the column in `df_imputed`. Confirm zero missing values remain.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
df_imputed = df.copy()
median_income = df_imputed["annual_income"].median()
df_imputed["annual_income"] = df_imputed["annual_income"].fillna(median_income)
print(df_imputed["annual_income"].isna().sum())

**Q13.** Do the same for `credit_score` in `df_imputed`: fill missing values with the column median. Confirm zero missing values remain.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
median_score = df_imputed["credit_score"].median()
df_imputed["credit_score"] = df_imputed["credit_score"].fillna(median_score)
print(df_imputed["credit_score"].isna().sum())

**Q14.** For a text column, a numeric median doesn't make sense. Fill missing `interest_rate` values in `df_imputed` with the literal string `"Unknown"` instead. Confirm zero missing values remain.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
df_imputed["interest_rate"] = df_imputed["interest_rate"].fillna("Unknown")
print(df_imputed["interest_rate"].isna().sum())

**Q15.** A single global median can be too crude — someone unemployed and someone with a high salary probably shouldn't get the same imputed income. On a **fresh copy** of the original `df` (`df_grouped_impute`), fill missing `annual_income` with the **median income for that person's own `employment_status`**, using `.groupby("employment_status")["annual_income"].transform("median")` to build a same-length "fill value" series, then `.fillna()` with it.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
income_by_employment = df.groupby("employment_status")["annual_income"].transform("median")
df_grouped_impute = df.copy()
df_grouped_impute["annual_income"] = df_grouped_impute["annual_income"].fillna(income_by_employment)
print(df_grouped_impute["annual_income"].isna().sum())

**Q16.** For time-ordered data, sometimes the most defensible fill is "carry the last known value forward." On a copy of `df` sorted by `application_date` (`df_ffill`), forward-fill missing `credit_score` values using `.ffill()`. Confirm zero missing values remain (as long as the very first row isn't missing — forward-fill has nothing to carry forward for a leading gap).

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
df_ffill = df.copy().sort_values("application_date")
df_ffill["credit_score"] = df_ffill["credit_score"].ffill()
print(df_ffill["credit_score"].isna().sum())

## Topic 14: Handling Duplicates

Duplicate rows quietly double-count customers and bias any statistic or model trained on them.

**Q17.** Use `.duplicated()` on the whole DataFrame to get a boolean Series flagging exact duplicate rows (`True` on the 2nd+ occurrence), into `is_dup`. Print how many rows are flagged with `.sum()`.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
is_dup = df.duplicated()
print(is_dup.sum())

**Q18.** `.duplicated()` by default only flags the *repeat* occurrences, not the original. Use `.duplicated(keep=False)` to flag **every** copy (original included) of a duplicated row, then filter `df` down to just those rows, into `full_duplicate_rows`. Print the shape — it should be roughly double the count from Q17.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
full_duplicate_rows = df[df.duplicated(keep=False)]
print(full_duplicate_rows.shape)

**Q19.** Remove exact duplicate rows with `.drop_duplicates()`, into `df_no_exact_dupes`. Print the shape — it should be `len(df)` minus your Q17 count.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
df_no_exact_dupes = df.drop_duplicates()
print(df_no_exact_dupes.shape)

**Q20.** A trickier case: the *same* `application_id` appearing more than once with *different* data (a re-submitted application). Count how many duplicate `application_id`s exist, into `dup_ids`, using `.duplicated()` on just that column.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
dup_ids = df["application_id"].duplicated().sum()
print(dup_ids)

**Q21.** Resolve that: drop duplicate `application_id`s, keeping the **last** occurrence (the most recent re-submission), using `.drop_duplicates(subset=["application_id"], keep="last")`, into `df_unique_ids`. Print the shape and confirm zero duplicate IDs remain.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
df_unique_ids = df.drop_duplicates(subset=["application_id"], keep="last")
print(df_unique_ids.shape, df_unique_ids["application_id"].duplicated().sum())

## Topic 15: Data Type Casting

`interest_rate` came in as text like `"5.72%"` — pandas can't do math on that until it's a real number.

**Q22.** Print the current dtype of `interest_rate`. Confirm it's text (`object`), not numeric.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
print(df["interest_rate"].dtype)

**Q23.** Strip the `%` character from every value in `interest_rate` using `.str.replace("%", "", regex=False)`, into `interest_rate_stripped`. Print the first 5 values — they should now look like plain number strings (missing/`"N/A"` values will still look odd for now).

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
interest_rate_stripped = df["interest_rate"].str.replace("%", "", regex=False)
print(interest_rate_stripped.head(5).tolist())

**Q24.** Convert `interest_rate_stripped` to actual numbers using `pd.to_numeric` with `errors="coerce"` (any value that can't be converted — like `"N/A"` or a missing value — becomes `NaN` instead of crashing the whole conversion), into `interest_rate_numeric`. Print its dtype and how many NaNs resulted.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
interest_rate_numeric = pd.to_numeric(interest_rate_stripped, errors="coerce")
print(interest_rate_numeric.dtype, interest_rate_numeric.isna().sum())

**Q25.** Put it together on a copy of `df` (`df_typed`): overwrite the `interest_rate` column with the cleaned, numeric version in one chained expression (strip `%`, then `pd.to_numeric` with `errors="coerce"`). Print the resulting dtype.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
df_typed = df.copy()
df_typed["interest_rate"] = pd.to_numeric(
    df_typed["interest_rate"].str.replace("%", "", regex=False), errors="coerce"
)
print(df_typed["interest_rate"].dtype)

**Q26.** `credit_score` is stored as `float64` only because missing values forced it out of integer form. Cast it to pandas' **nullable integer** type, `"Int64"` (capital I — this is different from numpy's plain `int64`, and it's the one that can hold `NaN` alongside whole numbers), on `df_typed`. Print the resulting dtype.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
df_typed["credit_score"] = df_typed["credit_score"].astype("Int64")
print(df_typed["credit_score"].dtype)

**Q27.** `loan_status` only ever takes a handful of repeated values (`"Approved"`/`"Denied"`/`"Pending"`) — a perfect fit for pandas' `"category"` dtype, which is far more memory-efficient than storing the same strings over and over. Cast it on `df_typed`. Print the resulting dtype.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
df_typed["loan_status"] = df_typed["loan_status"].astype("category")
print(df_typed["loan_status"].dtype)

## ✅ Checkpoint

**What you covered:**
- Identifying missing data: `.isnull()`/`.isna()`, per-column and total counts, missing **percentages**, and finding rows with any missing value
- Dropping missing data: `.dropna()` with `axis=1, how="all"` for empty columns, `subset=` for targeted row drops, `thresh=` for a forgiving rule, and *why* dropping empty columns has to come before dropping incomplete rows
- Imputing missing data: `.fillna()` with a global median, a **grouped** median via `.groupby().transform()`, a placeholder string for text, and `.ffill()` for time-ordered data
- Handling duplicates: `.duplicated()` vs `.duplicated(keep=False)`, `.drop_duplicates()`, and targeting duplicates on a business key (`application_id`) rather than the whole row
- Data type casting: `.str.replace()` + `pd.to_numeric(errors="coerce")` to turn messy text into numbers, `"Int64"` (nullable integer) vs numpy `int64`, and `"category"` for repeated text

**Why it matters for the project:** this phase turns the raw, messy export into something you can actually trust for analysis — every later phase assumes the data has already been through steps like these.

**What's next:** Phase 4, whenever you're ready — let me know the topics you want covered.